<a href="https://colab.research.google.com/github/aryan-param-69/Aryan-Health-Tracker/blob/main/Aryan_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question 8

Split a long tuple t into equal chunks of size k as a tuple of tuples. If the last chunk is short, include it as is.

Normal Method

In [2]:
def split_tuple(t, k):
    result = []

    for i in range(0, len(t), k):
        result.append(t[i:i+k])

    return tuple(result)


# Example
t = (1, 2, 3, 4, 5, 6, 7)
k = 3

print(split_tuple(t, k))

((1, 2, 3), (4, 5, 6), (7,))


Comprehension Method

Question 23

Find the longest contiguous strictly increasing run in a list. Return the sublist.

Normal Method

In [5]:
def longest_increasing_run(lst):

    max_run = []
    current_run = [lst[0]]

    for i in range(1, len(lst)):
        if lst[i] > lst[i - 1]:
            current_run.append(lst[i])
        else:
            if len(current_run) > len(max_run):
                max_run = current_run
            current_run = [lst[i]]

    if len(current_run) > len(max_run):
        max_run = current_run

    return max_run


# Example
nums = [1, 2, 3, 1, 2, 8, 9, 3]

print(longest_increasing_run(nums))

[1, 2, 8, 9]


Comprhensive method

Question 38

Given a mapping old→new, transform keys of data dict accordingly. If a key is not in mapping leave it as is. Handle collisions by summing values.

Normal Method

In [6]:
def replace_keys(data, mapping):

    result = {}

    for key, value in data.items():

        if key in mapping:
            new_key = mapping[key]
        else:
            new_key = key

        if new_key in result:
            result[new_key] += value
        else:
            result[new_key] = value

    return result


# Example
data = {"a": 10, "b": 20, "c": 5}
mapping = {"a": "x", "b": "x"}

print(replace_keys(data, mapping))

{'x': 30, 'c': 5}


Comprehension Method (using defaultdict)

from collections import defaultdict


In [ ]:
def replace_keys(data, mapping):

    result = defaultdict(int)

    for k,v in data.items():
        result[mapping.get(k,k)] += v

    return dict(result)

Question 53

From a list of emails extract a set of unique domains (case-insensitive, strip subaddressing like +tag if present).

Normal Method

In [7]:
def unique_domains(emails):

    domains = set()

    for email in emails:

        parts = email.split('@')
        local = parts[0]
        domain = parts[1]

        local = local.split('+')[0]

        domains.add(domain.lower())

    return domains


# Example
emails = [
    "aryan@gmail.com",
    "alice+work@yahoo.com",
    "bob@GMAIL.com",
    "test+spam@yahoo.com"
]

print(unique_domains(emails))

{'gmail.com', 'yahoo.com'}


Comprehension Method

In [ ]:
def unique_domains(emails):

    return {email.split('@')[1].lower() for email in emails}

 reasoning question 1&6

In [9]:
d1 = {True: 5, 2: 3}
d2 = {1: 10, False: 7}

canonical = lambda k: (type(k).__name__, k)

merged = {
    canonical(k): v
    for d in [d1, d2]
    for k, v in d.items()
}

print(merged)

{('bool', True): 5, ('int', 2): 3, ('int', 1): 10, ('bool', False): 7}


In [10]:
rows = [
    [1, 2, 2, 3],
    [3, 1, 2],
    [4, 5],
    [5, 4, 4],
    [6]
]

seen = set()
deduped_rows = []

for row in rows:
    row_set = frozenset(row)

    if row_set not in seen:
        seen.add(row_set)
        deduped_rows.append(list(row_set))

print(deduped_rows)

[[1, 2, 3], [4, 5], [6]]


LAB-1 log_analyzer.py

In [11]:
from collections import defaultdict, Counter, deque
import heapq
from datetime import datetime

# ---------------------------
# Parse log line
# ---------------------------
def parse_line(line):
    ts, endpoint, status = [x.strip() for x in line.split(",")]
    dt = datetime.fromisoformat(ts.replace("Z",""))
    hour = dt.strftime("%Y-%m-%dT%H")
    return hour, int(status)


# ---------------------------
# Median (pure python)
# ---------------------------
def median(values):
    s = sorted(values)
    n = len(s)

    if n == 0:
        return 0

    mid = n // 2

    if n % 2 == 0:
        return (s[mid-1] + s[mid]) / 2
    else:
        return s[mid]


# ---------------------------
# Median Absolute Deviation
# ---------------------------
def mad(values):
    m = median(values)
    deviations = [abs(v - m) for v in values]
    return median(deviations)


# ---------------------------
# Update streaming state
# ---------------------------
def update_state(counts, hour, code):
    counts[hour][code] += 1


# ---------------------------
# Top K status codes
# ---------------------------
def top_k(counter, k=3):
    return heapq.nlargest(k, counter.items(), key=lambda x: x[1])


# ---------------------------
# Burst detection
# ---------------------------
def detect_bursts(hour_order, counts, W=5):
    bursts = []

    for i, hour in enumerate(hour_order):

        window_hours = hour_order[max(0, i-W):i]

        for code, value in counts[hour].items():

            history = [
                counts[h].get(code,0)
                for h in window_hours
            ]

            if len(history) < 2:
                continue

            med = median(history)
            m = mad(history)

            threshold = med + 3*m

            if value > threshold:
                bursts.append((hour, code, value, threshold))

    return bursts


# ---------------------------
# Streaming processor
# ---------------------------
def process_log(file_path, k=3, W=5):

    counts = defaultdict(Counter)
    hour_order = []

    with open(file_path) as f:

        for line in f:

            hour, code = parse_line(line)

            if hour not in counts:
                hour_order.append(hour)

            update_state(counts, hour, code)

    # top-k per hour
    hourly_topk = {
        hour: top_k(counter, k)
        for hour, counter in counts.items()
    }

    bursts = detect_bursts(hour_order, counts, W)

    return hourly_topk, bursts


# ---------------------------
# Example run
# ---------------------------
if __name__ == "__main__":

    topk, bursts = process_log("server.log")

    print("\nTop-K status codes per hour\n")

    for hour, values in topk.items():
        print(hour, values)

    print("\nBurst detection\n")

    for b in bursts:
        print(b)

FileNotFoundError: [Errno 2] No such file or directory: 'server.log'

tests_log_analyzer.py

In [12]:
import unittest
from log_analyzer import parse_line, median, mad


class TestAnalyzer(unittest.TestCase):

    def test_parse(self):
        line = "2026-02-26T10:15:03Z, /api/login, 200"
        hour, code = parse_line(line)

        self.assertEqual(hour, "2026-02-26T10")
        self.assertEqual(code, 200)

    def test_median(self):
        self.assertEqual(median([1, 2, 3]), 2)
        self.assertEqual(median([1, 2, 3, 4]), 2.5)

    def test_mad(self):
        self.assertEqual(mad([1, 1, 2, 2, 4]), 1)


if __name__ == "__main__":
    unittest.main(argv=[''], exit=False)

ModuleNotFoundError: No module named 'log_analyzer'